# Практика · Word2Vec

> Лекція: [lecture.html](lecture.html) · Домашнє завдання: [homework.html](homework.html) ·
> Тест: [quiz.html](quiz.html)

Тут ми рахуємо **всі** числа, які називає лекція, і в тому самому порядку.

Що зробимо:

1. Зберемо корпус українських перекладів і наріжемо з нього пари «слово — сусід».
2. Порахуємо, у скільки разів повний `softmax` дорожчий за негативне семплювання.
3. Напишемо skip-gram із негативним семплюванням **своїми руками** на `torch`
   і звіримо власноруч виведений градієнт з автоматичним — до чотирнадцятого знака.
4. Навчимо модель тричі, з зернами 0, 1 і 2, і назвемо процесорний час.
5. Подивимось на сусідів і перевіримо, наскільки вони стійкі між зернами.
6. Заміряємо головне число блоку: **розрив** між синонімами й випадковими парами
   проти того самого розриву в TF-IDF.
7. Навчимо CBOW і чесно порівняємо два напрямки передбачення.

> ⏱ Заміряно: **близько трьох хвилин процесорного часу** на одному ядрі без
> відеокарти (`OMP_NUM_THREADS=1`). Найдорожче — три навчання skip-gram по 37 секунд
> кожне. Стінний час залежить від того, чим ще зайнята машина: у наших прогонах
> він виходив від 170 до 280 секунд на той самий зошит. Саме тому всі числа часу
> нижче — процесорні.

## 0 · Спершу фіксуємо потоки

Це не косметика. Якщо не обмежити OpenMP одним потоком, потоки крутяться в
очікуванні — і це очікування рахується як робота. Той самий замір роздувається
в десятки разів, а сам зошит іде вчетверо довше.

Змінні середовища мусять стояти **до** імпорту `numpy` і `torch`: після імпорту
бібліотека вже прочитала їх і не перечитує.

In [ ]:
import os
# ⚠️ обовʼязково ДО імпорту numpy і torch, інакше замір часу бреше
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

import sys, re, glob, gettext, time, math, collections, warnings
import numpy as np
import torch
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings('ignore')      # прибираємо шум чужих попереджень
torch.set_num_threads(1)               # той самий запобіжник, але для torch

print("Python  ", sys.version.split()[0])
print("numpy   ", np.__version__)
print("torch   ", torch.__version__)
print("sklearn ", sklearn.__version__)
print("потоків torch:", torch.get_num_threads())

## 1 · Корпус

Той самий, що в усьому курсі: українські переклади інтерфейсів із файлів `.mo`
системної локалі. Кожен запис — трійка «програма, англійський оригінал,
український переклад».

Якщо української локалі на машині немає, беремо вбудований мінікорпус. Тоді всі
числа будуть інші — і зошит про це прямо скаже.

In [ ]:
FALLBACK = [
    # (англійський оригінал, переклад А, переклад Б) — два різні переклади того
    # самого рядка. Саме такі пари потрібні нам для головного заміру теми.
    ("Failed to open the configuration file",
     "Не вдалося відкрити файл конфігурації для читання",
     "Помилка доступу до налаштувань програми при відкриванні"),
    ("Not enough space on the disk",
     "Не вдалося записати файл на диск: недостатньо місця",
     "Бракує вільного простору у сховищі для збереження даних"),
    ("Cannot create a temporary file",
     "Помилка під час створення тимчасового файла у теці кешу",
     "Не вдалося зробити проміжний запис у каталозі обміну"),
    ("Choose the output folder",
     "Виберіть теку, у яку буде збережено вихідний файл",
     "Вкажіть каталог призначення для результату роботи"),
    ("Connection to the server failed",
     "Не вдалося встановити зʼєднання з сервером за вказаною адресою",
     "Помилка звʼязку з вузлом мережі: адресу недоступно"),
    ("The server closed the connection",
     "Сервер розірвав зʼєднання під час передавання даних",
     "Віддалений вузол припинив обмін інформацією достроково"),
    ("Request timed out",
     "Перевищено час очікування відповіді від віддаленого сервера",
     "Вузол не надіслав жодних даних у відведений проміжок"),
    ("Host not found",
     "Не вдалося визначити адресу вузла у локальній мережі",
     "Імʼя компʼютера відсутнє у службі пошуку сегмента"),
    ("Enter your account password",
     "Введіть пароль облікового запису для доступу до ресурсу",
     "Наберіть таємне слово користувача, щоб відкрити сховище"),
    ("The password is too short",
     "Пароль надто короткий: потрібно щонайменше вісім символів",
     "Таємне слово має містити більше літер, ніж ви набрали"),
    ("Cannot verify the password",
     "Не вдалося перевірити пароль: обліковий запис заблоковано",
     "Звірити таємне слово неможливо, бо користувача вимкнено"),
    ("The settings dialog cannot be opened",
     "Діалогове вікно налаштувань не вдалося відкрити повторно",
     "Панель параметрів програми більше не показується на екрані"),
    ("Close the current window",
     "Закрити поточне вікно і повернутися до головного екрана",
     "Згорнути активну панель та показати основну сторінку"),
    ("The window size will be remembered",
     "Розмір вікна буде збережено між запусками програми",
     "Габарити панелі лишаться такими самими після перезавантаження"),
    ("Press the button to apply",
     "Натисніть кнопку, щоб застосувати вибрані налаштування",
     "Клацніть перемикач для збереження заданих параметрів"),
    ("The button stays disabled",
     "Кнопка залишається неактивною, доки не вибрано жодного пристрою",
     "Перемикач не працює, поки не позначено жодного обладнання"),
    ("Choose the background colour",
     "Виберіть колір тла для вибраного шару зображення",
     "Вкажіть відтінок підкладки позначеної частини малюнка"),
    ("Fill opacity is set in percent",
     "Прозорість заливки задається у відсотках від повного кольору",
     "Щільність фарбування вимірюється часткою насиченого відтінку"),
    ("Cannot load the font",
     "Не вдалося завантажити шрифт із вказаного каталогу",
     "Помилка читання накреслення літер у заданій теці"),
    ("Font size restored",
     "Розмір шрифту змінено на значення за замовчуванням",
     "Висоту літер повернуто до початкової величини"),
    ("The device was disconnected",
     "Пристрій відключено від системної шини під час запису",
     "Обладнання зникло з каналу обміну просто посеред збереження"),
    ("The block device is not ready",
     "Блоковий пристрій не готовий до читання даних",
     "Носій інформації ще не може віддавати вміст"),
    ("An internal error occurred",
     "Виникла внутрішня помилка під час обробки запиту",
     "Сталася критична відмова всередині служби опрацювання"),
    ("The application will be closed",
     "Критична помилка застосунку: роботу буде завершено",
     "Серйозна відмова програми: сеанс примусово припиняється"),
    ("Cannot delete the selected record",
     "Сталася помилка під час видалення позначеного запису",
     "Вилучити відмічений рядок бази не вдалося через відмову"),
    ("Invalid value format",
     "Некоректний формат значення у полі введення",
     "Неправильний вигляд числа в комірці заповнення"),
    ("The value must be a positive integer",
     "Значення параметра має бути цілим додатним числом",
     "Число у полі мусить бути натуральним і більшим за нуль"),
    ("Backup failed",
     "Не вдалося створити резервну копію бази даних",
     "Помилка збереження запасного примірника сховища"),
    ("The database is read-only",
     "Базу даних відкрито у режимі лише для читання",
     "Сховище доступне без права змінювати його вміст"),
    ("A file with this name already exists",
     "Каталог призначення вже містить файл із такою назвою",
     "У теці отримання є документ з таким самим імʼям"),
    ("The input file is damaged",
     "Вхідний файл пошкоджено або він має невідомий формат",
     "Отриманий документ зіпсовано чи його вигляд невідомий"),
    ("Executable not found",
     "Виконуваний файл не знайдено у вказаному каталозі",
     "Програму для запуску не знайдено у заданій теці"),
    ("Cannot change permissions",
     "Не вдалося змінити права доступу до тимчасового каталогу",
     "Дозволи для проміжної теки лишилися без змін через відмову"),
    ("Sound is muted for this session",
     "Звук вимкнено для всіх застосунків цього сеансу",
     "Гучність приглушено в кожній програмі поточного входу"),
    ("Volume will be restored",
     "Гучність звуку буде відновлено після перезапуску служби",
     "Рівень сигналу повернеться, коли перезавантажиться служба"),
    ("Cannot play the sound",
     "Не вдалося відтворити звук: пристрій виводу зайнято",
     "Програти сигнал неможливо, бо обладнання виведення працює"),
    ("The network interface is not configured",
     "Мережевий інтерфейс не налаштовано для автоматичного запуску",
     "Плату звʼязку не підготовлено до самостійного вмикання"),
    ("Check the network settings",
     "Перевірте налаштування мережі та повторіть спробу",
     "Звірте параметри звʼязку і спробуйте виконати дію ще раз"),
    ("Cannot reach the update service",
     "Не вдалося отримати дані від служби оновлення системи",
     "Звернутися до сервісу свіжих версій операційної системи не вийшло"),
    ("Restart is required",
     "Оновлення встановлено, потрібне перезавантаження системи",
     "Свіжу версію записано; компʼютер слід запустити наново"),
]

def load_system_corpus():
    """Трійки (програма, англійський оригінал, український переклад) із .mo-файлів."""
    docs = []
    for path in sorted(glob.glob('/usr/share/locale/uk/LC_MESSAGES/*.mo')):
        try:
            with open(path, 'rb') as handle:
                catalog = gettext.GNUTranslations(handle)
        except Exception:
            continue                    # чужий або зламаний формат — просто пропускаємо
        program = path.split('/')[-1][:-3]
        for source, target in catalog._catalog.items():
            # службовий заголовок каталогу має ключ '' і текстом не є
            if isinstance(source, str) and isinstance(target, str) \
               and len(target) > 30 and 'Project-Id' not in target:
                docs.append((program, source, target))
    return docs

corpus = load_system_corpus()
REAL_CORPUS = len(corpus) > 5000
if not REAL_CORPUS:
    print("⚠️ української локалі на цій машині немає — беремо мінікорпус.")
    print("   Зошит виконається весь, але числа будуть інші, ніж у лекції.")
    corpus = []
    for source, first, second in FALLBACK:
        corpus.append(("fallback", source, first))
        corpus.append(("fallback", source, second))

documents = [target for _, _, target in corpus]
print("документів :", len(documents))
print("програм    :", len({program for program, _, _ in corpus}))
print("приклад    :", documents[17])

## 2 · Токенізатор і словник моделі

Токенізатор — канонічний для всього курсу: українські літери, апостроф працює як
звʼязка всередині слова, а не як окремий символ.

Слова, що трапились рідше за десять разів, у словник моделі не потрапляють. Причина
не в економії памʼяті: на трьох-чотирьох прикладах вектор нема з чого вивести, і
такий вектор лишиться майже випадковим — але шумітиме в усіх обчисленнях.

In [ ]:
TOKEN_PATTERN = r"[а-яїієґ]+(?:['ʼ’][а-яїієґ]+)*"
tokenize = re.compile(TOKEN_PATTERN).findall

MIN_COUNT = 10 if REAL_CORPUS else 2   # скільки разів слово має трапитись, щоб дістати вектор
WINDOW = 5              # скільки сусідів ліворуч і праворуч рахуємо контекстом
DIM = 64                # довжина вектора слова
NEGATIVE = 5            # скільки «неправильних» слів на один правильний
BATCH = 4096
EPOCHS = 2
SEEDS = (0, 1, 2)

tokenized = [tokenize(text.lower()) for text in documents]
frequency = collections.Counter()
for words in tokenized:
    frequency.update(words)

vocabulary = [word for word, count in frequency.most_common() if count >= MIN_COUNT]
word_id = {word: i for i, word in enumerate(vocabulary)}
total_tokens = sum(frequency.values())
covered = sum(frequency[word] for word in vocabulary)

print(f"слововживань      : {total_tokens}")
print(f"різних словоформ  : {len(frequency)}")
print(f"словник моделі    : {len(vocabulary)} слів при min_count={MIN_COUNT}")
print(f"покриття тексту   : {100 * covered / total_tokens:.4f} %")
print(f"найчастіші        : {', '.join(vocabulary[:8])}")

## 3 · Вікно контексту: скільки прикладів дає текст

Skip-gram бере слово в центрі й кожного його сусіда в межах вікна — і робить із
цієї пари один навчальний приклад. Порахуймо, скільки таких пар дає наш корпус і
наскільки вікно взагалі буває заповнене.

Це важливо саме для нашого корпусу: документи в ньому короткі, тож «вікно ±5» —
радше обіцянка, ніж факт.

In [ ]:
# кожен документ перетворюємо на послідовність номерів слів,
# викидаючи те, чого немає в словнику моделі
sequences = [np.array([word_id[w] for w in words if w in word_id], dtype=np.int32)
             for words in tokenized]
sequences = [s for s in sequences if len(s) >= 2]      # з одного слова пари не зробиш

def build_pairs(sequences, window):
    """Усі впорядковані пари (центр, сусід) у межах вікна.

    Замість подвійного циклу зсуваємо послідовність на 1..window позицій:
    так пара «центр і сусід через k слів» отримується одним зрізом.
    """
    centers, contexts = [], []
    for sequence in sequences:
        length = len(sequence)
        for offset in range(1, window + 1):
            if offset >= length:
                break
            centers.append(sequence[:-offset]);  contexts.append(sequence[offset:])
            centers.append(sequence[offset:]);   contexts.append(sequence[:-offset])
    return np.concatenate(centers), np.concatenate(contexts)

pair_center, pair_context = build_pairs(sequences, WINDOW)
n_positions = sum(len(s) for s in sequences)

print(f"позицій-центрів        : {n_positions}")
print(f"пар при вікні ±{WINDOW}       : {len(pair_center)}")
print(f"сусідів на центр       : {len(pair_center) / n_positions:.4f} із можливих {2 * WINDOW}")

full = 0
for sequence in sequences:
    positions = np.arange(len(sequence))
    width = np.minimum(positions + WINDOW, len(sequence) - 1) - np.maximum(positions - WINDOW, 0)
    full += int(np.sum(width == 2 * WINDOW))
print(f"центрів із повним вікном: {full} — це {100 * full / n_positions:.1f} % позицій")

Тепер те саме для менших вікон. Число пар — це і є ціна навчання: вдвічі більше
пар означає вдвічі довше навчання.

In [ ]:
print(f"{'вікно':>6} {'пар':>12} {'приріст':>10}")
previous = None
for window in range(1, WINDOW + 1):
    total = 0
    for sequence in sequences:
        positions = np.arange(len(sequence))
        width = np.minimum(positions + window, len(sequence) - 1) - np.maximum(positions - window, 0)
        total += int(width.sum())
    growth = "—" if previous is None else f"+{100 * (total / previous - 1):.1f} %"
    print(f"{window:>6} {total:>12} {growth:>10}")
    previous = total

Вікно виросло впʼятеро, а пар стало більше лише втричі: у коротких документах
розширювати вікно нема куди. Це властивість корпусу, а не алгоритму — і саме тому
її треба заміряти, а не припускати.

## 4 · Чому не можна просто взяти softmax

Класична постановка: за центральним словом передбачити сусіда як **один клас із
усього словника**. Щоб порахувати ймовірність, softmax мусить перебрати весь
словник — на кожному прикладі.

Порахуймо ціну обох варіантів у множеннях.

In [ ]:
def softmax_cost(vocab_size, dim):
    """Скалярний добуток центрального вектора з КОЖНИМ словом словника."""
    return vocab_size * dim

def negative_cost(negatives, dim):
    """Скалярний добуток лише з правильним словом і з кількома неправильними."""
    return (1 + negatives) * dim

full_softmax = softmax_cost(len(vocabulary), DIM)
sampled = negative_cost(NEGATIVE, DIM)

print(f"словник моделі {len(vocabulary)} слів, вимірів {DIM}, негативних {NEGATIVE}")
print(f"  повний softmax     : {full_softmax:>10} множень на приклад")
print(f"  негативне семплюв. : {sampled:>10} множень на приклад")
print(f"  дешевше у          : {full_softmax / sampled:>10.1f} раза")
print()
print(f"{'словник':>10} {'softmax':>14} {'семплювання':>13} {'дешевше у':>12}")
for vocab_size in (6074, 27078, 100000, 1000000):
    a, b = softmax_cost(vocab_size, DIM), sampled
    print(f"{vocab_size:>10} {a:>14} {b:>13} {a / b:>11.1f}x")
print()
print(f"усього пар {len(pair_center)} x {EPOCHS} епохи = {len(pair_center) * EPOCHS} прикладів;")
print(f"це {len(pair_center) * EPOCHS * full_softmax / 1e12:.2f} трлн множень із softmax "
      f"проти {len(pair_center) * EPOCHS * sampled / 1e9:.1f} млрд без нього")

## 5 · Звідки беруться «неправильні» слова

Негативні приклади тягнуть не рівномірно й не за самою частотою, а за частотою в
степені **0.75**. Подивімось, що ця дрібна на вигляд деталь робить із розподілом.

In [ ]:
word_count = np.zeros(len(vocabulary), dtype=np.int64)
for sequence in sequences:
    np.add.at(word_count, sequence, 1)          # рахуємо лише те, що дійсно потрапило у вікна

# найрідкісніше слово словника — останнє в ньому: словник упорядкований за частотою
rare_index = len(vocabulary) - 1
rarest = int(word_count[rare_index])
print(f"найчастіше слово словника  : «{vocabulary[0]}» — {word_count[0]} разів")
print(f"найрідкісніше слово словника: «{vocabulary[rare_index]}» — {rarest} разів")
print(f"відношення частот: {word_count[0] / rarest:.1f}x")
print()
print(f"{'степінь':>8} {'не':>9} {'для':>9} {'файл':>9} {'найрідкісніше':>15} {'перекіс':>10}")
for alpha in (0.0, 0.5, 0.75, 1.0):
    weights = word_count.astype(float) ** alpha
    probability = weights / weights.sum()
    row = [f"{100 * probability[word_id[w]]:.3f} %" if w in word_id else "—"
           for w in ('не', 'для', 'файл')]
    skew = probability[0] / probability[rare_index]
    print(f"{alpha:>8} {row[0]:>9} {row[1]:>9} {row[2]:>9} "
          f"{100 * probability[rare_index]:>13.5f} % {skew:>9.1f}x")

NEGATIVE_ALPHA = 0.75
sampling_weights = word_count.astype(np.float64) ** NEGATIVE_ALPHA
sampling_weights /= sampling_weights.sum()
sampling_cumulative = np.cumsum(sampling_weights)

Степінь 0.75 — не магія й не результат виведення: це число дібрали
експериментально автори Word2Vec — Mikolov та ін., 2013, розділ 2.2 *(цитата з
літератури, а не наш замір)*. Наш замір
показує, **що саме** воно робить: перекіс між найчастішим і найрідкіснішим словом
падає з 2792 разів до 384, але не зникає до одиниці.

## 6 · Модель: дві матриці й одна логістична функція

Кожне слово має **два** вектори: один як центральне слово, другий як контекст.
Оцінка пари — їхній скалярний добуток. Далі логістична функція перетворює будь-яке
число на ймовірність від нуля до одиниці, а функція втрат каже: для справжньої пари
ця ймовірність має бути близька до одиниці, для вигаданої — до нуля.

In [ ]:
class SkipGramNegativeSampling(torch.nn.Module):
    """Skip-gram із негативним семплюванням, написаний повністю руками."""

    def __init__(self, vocab_size, dim):
        super().__init__()
        # вектори слова в ролі центрального — саме їх ми потім і використовуємо
        self.center = torch.nn.Embedding(vocab_size, dim)
        # вектори того самого слова в ролі контексту — робочі, для навчання
        self.context = torch.nn.Embedding(vocab_size, dim)
        # маленькі випадкові центральні й рівно нульові контекстні:
        # так на початку кожна оцінка дорівнює нулю, тобто ймовірність рівно 0.5
        torch.nn.init.uniform_(self.center.weight, -0.5 / dim, 0.5 / dim)
        torch.nn.init.zeros_(self.context.weight)

    def scores(self, centers, contexts, negatives):
        v = self.center(centers)                       # (B, DIM) — центральні вектори
        u_positive = self.context(contexts)            # (B, DIM) — справжній сусід
        u_negative = self.context(negatives)           # (B, NEG, DIM) — вигадані сусіди
        positive = (v * u_positive).sum(dim=1)                       # (B,)
        negative = torch.bmm(u_negative, v.unsqueeze(2)).squeeze(2)  # (B, NEG)
        return positive, negative


def sgns_loss(positive, negative):
    """Втрата: справжню пару тягнемо до 1, кожну вигадану — до 0.

    logsigmoid замість log(sigmoid(...)) — щоб не втратити точність
    на великих за модулем оцінках.
    """
    good = torch.nn.functional.logsigmoid(positive).mean()
    bad = torch.nn.functional.logsigmoid(-negative).sum(dim=1).mean()
    return -(good + bad)


def draw_negatives(rng, size):
    """Випадкові слова за розподілом «частота у степені 0.75»."""
    return np.searchsorted(sampling_cumulative, rng.random(size)).astype(np.int64)

print("модель описано; параметрів у ній:",
      2 * len(vocabulary) * DIM, f"({len(vocabulary)} слів x {DIM} вимірів x 2 матриці)")

## 7 · Перевірка: наш градієнт = автоматичний

Найцінніше в практиці — переконатись, що всередині немає магії. Виведімо похідну
втрати по центральному вектору руками й порівняймо з тим, що дає `autograd`.

Для однієї справжньої пари з оцінкою `s` похідна `−log σ(s)` по `s` дорівнює
`σ(s) − 1`, а для вигаданої пари похідна `−log σ(−s)` дорівнює `σ(s)`. Далі
залишається помножити на відповідний контекстний вектор.

In [ ]:
torch.manual_seed(0)
check = SkipGramNegativeSampling(len(vocabulary), DIM)
rng_check = np.random.default_rng(0)
# невеличкий батч, щоб порівняння читалось
sample = rng_check.choice(len(pair_center), 64)
c = torch.from_numpy(pair_center[sample].astype(np.int64))
o = torch.from_numpy(pair_context[sample].astype(np.int64))
neg = torch.from_numpy(draw_negatives(rng_check, (64, NEGATIVE)))

# центральні вектори роблять ненульовими, бо нульові приховали б помилку
with torch.no_grad():
    check.context.weight.uniform_(-0.3, 0.3)

positive, negative = check.scores(c, o, neg)
loss = sgns_loss(positive, negative)
check.zero_grad(); loss.backward()
auto_gradient = check.center.weight.grad.detach().numpy()

# те саме руками, у numpy
V = check.center.weight.detach().numpy()
U = check.context.weight.detach().numpy()
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

manual = np.zeros_like(V)
centers_np, contexts_np, negatives_np = c.numpy(), o.numpy(), neg.numpy()
batch_size = len(centers_np)
for row in range(batch_size):
    center_vector = V[centers_np[row]]
    score_positive = center_vector @ U[contexts_np[row]]
    # похідна для справжньої пари; ділимо на batch_size, бо у втраті стоїть mean
    manual[centers_np[row]] += (sigmoid(score_positive) - 1.0) * U[contexts_np[row]] / batch_size
    for negative_word in negatives_np[row]:
        score_negative = center_vector @ U[negative_word]
        manual[centers_np[row]] += sigmoid(score_negative) * U[negative_word] / batch_size

difference = float(np.max(np.abs(manual - auto_gradient)))
assert np.allclose(manual, auto_gradient, atol=1e-6), "градієнт розійшовся!"
print(f"✅ збігається: найбільша різниця між ручним і автоматичним градієнтом {difference:.2e}")

## 8 · Перевірка відтворюваності

Три зерна показують розкид по **даних**, але не бачать, чи взагалі відтворюється
сам прогін. Перевіряємо це окремо: два навчання з тим самим зерном на тих самих
даних мають дати побітово однакові ваги.

In [ ]:
def tiny_run(seed, steps=30):
    """Коротке навчання на кількох батчах — рівно для перевірки відтворюваності."""
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    model = SkipGramNegativeSampling(len(vocabulary), DIM)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    for _ in range(steps):
        batch = rng.choice(len(pair_center), 512)
        positive, negative = model.scores(
            torch.from_numpy(pair_center[batch].astype(np.int64)),
            torch.from_numpy(pair_context[batch].astype(np.int64)),
            torch.from_numpy(draw_negatives(rng, (512, NEGATIVE))))
        loss = sgns_loss(positive, negative)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    return model.center.weight.detach().numpy().copy()

first, second, other = tiny_run(0), tiny_run(0), tiny_run(1)
assert np.array_equal(first, second), "той самий прогін дав інший результат!"
print("✅ два прогони з зерном 0 збіглися побітово")
print(f"   а зерно 1 дає інші ваги: найбільша різниця {np.max(np.abs(first - other)):.4f}")

## 9 · Навчання skip-gram: три зерна

Тепер повне навчання. Дві епохи по всіх парах, Adam зі швидкістю навчання 0.01,
батч 4096. Час міряємо **процесорний**, бо машина може бути зайнята чужою роботою.

In [ ]:
def train_skipgram(seed):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    model = SkipGramNegativeSampling(len(vocabulary), DIM)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

    started_cpu = time.process_time()
    started_wall = time.perf_counter()
    curve, running, steps = [], 0.0, 0
    for epoch in range(EPOCHS):
        order = rng.permutation(len(pair_center))       # перемішуємо пари щоепохи
        for start in range(0, len(order), BATCH):
            batch = order[start:start + BATCH]
            positive, negative = model.scores(
                torch.from_numpy(pair_center[batch].astype(np.int64)),
                torch.from_numpy(pair_context[batch].astype(np.int64)),
                torch.from_numpy(draw_negatives(rng, (len(batch), NEGATIVE))))
            loss = sgns_loss(positive, negative)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            running += loss.item(); steps += 1
            if steps % 50 == 0:
                curve.append(running / 50); running = 0.0
    return {
        'center':  model.center.weight.detach().numpy().copy(),
        'context': model.context.weight.detach().numpy().copy(),
        'cpu':     time.process_time() - started_cpu,
        'wall':    time.perf_counter() - started_wall,
        'curve':   curve,
        'steps':   steps,
    }

skipgram = {}
for seed in SEEDS:
    skipgram[seed] = train_skipgram(seed)
    result = skipgram[seed]
    print(f"зерно {seed}: {result['steps']} кроків, процесорних {result['cpu']:.1f} с "
          f"(стінних {result['wall']:.1f} с), втрата {result['curve'][0]:.4f} -> {result['curve'][-1]:.4f}")

cpu_times = np.array([skipgram[s]['cpu'] for s in SEEDS])
print(f"\nнавчання skip-gram: {cpu_times.mean():.1f} ±{cpu_times.std():.1f} с процесорних")

## 10 · Що вивчилось: найближчі сусіди

Косинусна близькість двох векторів — це косинус кута між ними: одиниця, якщо
дивляться в один бік, нуль, якщо перпендикулярні. Щоб її порахувати, досить
поділити кожен вектор на його довжину й перемножити.

In [ ]:
def unit_rows(matrix):
    """Кожен рядок ділимо на його довжину — після цього скалярний добуток і є косинус."""
    return matrix / np.linalg.norm(matrix, axis=1, keepdims=True)

def neighbours(unit_matrix, word, count=4):
    if word not in word_id:
        return []
    similarity = unit_matrix @ unit_matrix[word_id[word]]
    order = np.argsort(-similarity)[1:count + 1]        # перший — саме слово
    return [(vocabulary[j], float(similarity[j])) for j in order]

PROBE = [word for word in ('файл', 'помилка', 'вікно', 'мережі') if word in word_id]
unit_center = unit_rows(skipgram[0]['center'])
print("сусіди за skip-gram, зерно 0:")
for word in PROBE:
    found = neighbours(unit_center, word)
    if found:
        print(f"{word:<9} -> " + ' · '.join(f"{w} {c:.3f}" for w, c in found))

Сусіди осмислені, і жодного добору тут немає — це просто чотири слова, які
приходять на думку першими для корпусу про програми.

Але перш ніж цитувати такий список, треба перевірити ще одну річ: чи він узагалі
той самий на іншому зерні.

In [ ]:
units = [unit_rows(skipgram[s]['center']) for s in SEEDS]
for word in PROBE:
    for seed, unit in zip(SEEDS, units):
        found = neighbours(unit, word)
        if found:
            print(f"  зерно {seed} · {word:<9} -> " + ' · '.join(w for w, _ in found))
    print()

def top_set(unit, word, count=10):
    similarity = unit @ unit[word_id[word]]
    return set(np.argsort(-similarity)[1:count + 1].tolist())

overlaps = []
for word in vocabulary[:min(500, len(vocabulary))]:   # 500 найчастіших слів словника
    sets = [top_set(unit, word) for unit in units]
    overlaps.append(len(sets[0] & sets[1]) / 10)
    overlaps.append(len(sets[0] & sets[2]) / 10)
    overlaps.append(len(sets[1] & sets[2]) / 10)
print(f"збіг топ-10 сусідів між зернами, 500 найчастіших слів: "
      f"{np.mean(overlaps):.4f} — тобто близько половини списку")

Ось перший неприємний, але чесний результат: **половина списку сусідів
змінюється від зерна до зерна**. Список — ілюстрація, а не замір. Числа, які
можна цитувати, мають бути усередненими по багатьох словах, і саме такі ми
рахуємо далі.

Перш ніж рухатись далі — подивімось на функцію втрат на справжніх числах
навченої моделі. Це ті самі два доданки, які лекція розбирає у розділі 05: справжня
пара має набрати багато, вигадана — мало.

In [ ]:
def logistic(x):
    return 1.0 / (1.0 + math.exp(-x))

center_matrix, context_matrix = skipgram[0]['center'], skipgram[0]['context']
print(f"{'пара':<22} {'добуток s':>10} {'сигмоїда':>10} {'втрата':>9}")
for first, second, kind in (('файл', 'вхідний', 'справжня'),
                            ('файл', 'тимчасовий', 'справжня'),
                            ('файл', 'звук', 'вигадана'),
                            ('файл', 'кольору', 'вигадана')):
    if first not in word_id or second not in word_id:
        continue
    score = float(center_matrix[word_id[first]] @ context_matrix[word_id[second]])
    probability = logistic(score)
    # для справжньої пари втрата −log σ(s), для вигаданої −log σ(−s)
    loss_value = -math.log(probability if kind == 'справжня' else 1 - probability)
    print(f"{first + ' · ' + second:<22} {score:>10.4f} {probability:>10.4f} {loss_value:>9.4f}")

## 11 · Головне число: розрив

Блок 2 закінчився стіною. Той самий англійський рядок різні перекладачі переклали
по-різному — і ми маємо пари документів, про які **точно** відомо, що вони означають
одне й те саме. TF-IDF дає таким парам косинус близько 0.22, а кожній пʼятій —
рівно нуль.

Тепер міряємо те саме на векторах. Правило чесності: порівнюємо не самі косинуси
синонімів, а **розрив** між синонімами й випадковими парами. Інакше метод, який
усьому підряд ставить 0.9, виглядав би переможцем.

In [ ]:
NOISE = re.compile(r"[@<>]|https?://")

def paraphrase_pairs():
    """Пари українських перекладів того самого англійського рядка,
    у яких майже немає спільних слів."""
    by_source = collections.defaultdict(dict)
    for program, source, target in corpus:
        if source.strip() == 'translator-credits':
            continue
        by_source[source][' '.join(target.split())] = program
    found = []
    for source, variants in by_source.items():
        texts = list(variants)
        for i in range(len(texts)):
            for j in range(i + 1, len(texts)):
                first, second = texts[i], texts[j]
                if NOISE.search(first) or NOISE.search(second):
                    continue
                a = set(tokenize(first.lower()))
                b = set(tokenize(second.lower()))
                if len(a) < 3 or len(b) < 3:
                    continue
                if len(a & b) / len(a | b) < 0.34:
                    found.append((first, second))
    return found

pairs = paraphrase_pairs()
print(f"пар «те саме іншими словами»: {len(pairs)}")
for first, second in pairs[:3]:
    print(f"    A: {first}")
    print(f"    B: {second}")

In [ ]:
def document_vector(text, matrix):
    """Вектор документа — середнє векторів його слів, зведене до довжини 1.

    Простіше не буває, і саме тому це чесна перевірка самих векторів:
    жодного навчання поверх них тут немає.
    """
    ids = [word_id[w] for w in tokenize(text.lower()) if w in word_id]
    if not ids:
        return None
    vector = matrix[ids].mean(axis=0)
    length = np.linalg.norm(vector)
    return vector / length if length > 0 else None

def cosines(matrix, left_texts, right_texts):
    out = []
    for left, right in zip(left_texts, right_texts):
        a, b = document_vector(left, matrix), document_vector(right, matrix)
        out.append(float(a @ b) if a is not None and b is not None else 0.0)
    return np.array(out)

def sample_documents(seed, n=20000):
    """Ті самі документи при тому самому зерні — як у темі 05."""
    rng = np.random.default_rng(seed)
    picked = rng.choice(len(documents), min(n, len(documents)), replace=False)
    return [documents[i] for i in picked]

def separation(synonym, random_pairs):
    """Частка випадків, коли пара синонімів набрала більше за випадкову пару."""
    return float(np.mean(synonym[:, None] > random_pairs[None, :])
                 + 0.5 * np.mean(synonym[:, None] == random_pairs[None, :]))

measurements = collections.defaultdict(list)
distributions = {}

for seed in SEEDS:
    texts = sample_documents(seed)
    rng = np.random.default_rng(7 + seed)
    left_random = [texts[i] for i in rng.choice(len(texts), len(pairs))]
    right_random = [texts[i] for i in rng.choice(len(texts), len(pairs))]

    # TF-IDF рахуємо тут-таки, щоб порівняння йшло на тих самих парах
    tfidf = TfidfVectorizer(token_pattern=TOKEN_PATTERN).fit(texts)
    left = tfidf.transform([p[0] for p in pairs])
    right = tfidf.transform([p[1] for p in pairs])
    tfidf_synonym = np.asarray(left.multiply(right).sum(axis=1)).ravel()
    a = tfidf.transform(left_random); b = tfidf.transform(right_random)
    tfidf_random = np.asarray(a.multiply(b).sum(axis=1)).ravel()
    measurements['TF-IDF'].append((tfidf_synonym.mean(), tfidf_random.mean(),
                                   separation(tfidf_synonym, tfidf_random),
                                   100 * np.mean(tfidf_synonym == 0)))
    if seed == 0:
        distributions['TF-IDF'] = (tfidf_synonym, tfidf_random)

    for name, matrix in (
            ('skip-gram · центральна', skipgram[seed]['center']),
            ('skip-gram · сума двох',  skipgram[seed]['center'] + skipgram[seed]['context'])):
        synonym = cosines(matrix, [p[0] for p in pairs], [p[1] for p in pairs])
        random_pairs = cosines(matrix, left_random, right_random)
        measurements[name].append((synonym.mean(), random_pairs.mean(),
                                   separation(synonym, random_pairs),
                                   100 * np.mean(synonym <= 0)))
        if seed == 0:
            distributions[name] = (synonym, random_pairs)

print(f"{'подання':<24} {'синоніми':>16} {'випадкові':>16} {'РОЗРИВ':>16} {'розділяє':>10}")
for name, rows in measurements.items():
    table = np.array(rows)
    gap = table[:, 0] - table[:, 1]
    print(f"{name:<24} {table[:,0].mean():>8.4f} ±{table[:,0].std():.4f} "
          f"{table[:,1].mean():>8.4f} ±{table[:,1].std():.4f} "
          f"{gap.mean():>8.4f} ±{gap.std():.4f} {table[:,2].mean():>9.4f}")

Читати цю таблицю треба уважно, і найголовніше тут — **не перший стовпчик**.

Ембединги піднімають косинус синонімів із 0.22 до 0.71. Але вони піднімають і
косинус випадкових пар — з 0.01 до 0.40. Якби ми цитували лише синоніми, вийшло б
«утричі краще»; чесна ж міра — розрив, і він виріс лише в півтора раза.

А от сума двох матриць міняє картину повністю: випадкові пари падають майже до
нуля, і розрив стає вдвічі з половиною більшим за TF-IDF. Чому — у наступному
розділі.

In [ ]:
base = np.array(measurements['TF-IDF'])
base_gap = (base[:, 0] - base[:, 1]).mean()
print(f"{'подання':<24} {'розрив':>10} {'проти TF-IDF':>14} {'нуль або відʼємні':>18}")
for name, rows in measurements.items():
    table = np.array(rows)
    gap = (table[:, 0] - table[:, 1]).mean()
    print(f"{name:<24} {gap:>10.4f} {gap / base_gap:>13.3f}x {table[:,3].mean():>14.1f} %")

## 12 · Дві матриці: чому сума працює краще

Кожне слово має центральний вектор і контекстний. Word2Vec за замовчуванням віддає
центральний, а контекстний викидає. Наш замір каже, що це не завжди найкраще
рішення — і причина проста й вимірна.

In [ ]:
def average_pairwise_cosine(matrix):
    """Середній косинус між усіма парами слів словника.

    Рахуємо без квадратної матриці: середнє попарних косинусів дорівнює
    квадрату довжини середнього одиничного вектора.
    """
    mean_unit = unit_rows(matrix).mean(axis=0)
    return float(mean_unit @ mean_unit)

for name, matrix in (('центральна', skipgram[0]['center']),
                     ('контекстна', skipgram[0]['context']),
                     ('сума двох',  skipgram[0]['center'] + skipgram[0]['context'])):
    print(f"{name:<12} середній косинус між усіма словами: {average_pairwise_cosine(matrix):.4f}")

Ось і пояснення. У центральній матриці **всі слова трохи схожі між собою**: є
спільний напрямок, у який дивляться всі вектори одразу. Він і завищує косинус
випадкових пар до 0.40.

Контекстна матриця має той самий дефект, тільки сильніший. А от у сумі спільні
напрямки двох матриць гасять один одного, і середній косинус падає майже до нуля.

Перевіримо це ще й прямо: приберемо в центральній матриці першу головну компоненту
(тобто саме той спільний напрямок) і подивимось, чи виросте розрив.

In [ ]:
def drop_first_component(matrix):
    """Прибираємо з векторів той єдиний напрямок, у який дивляться всі одразу."""
    centred = matrix - matrix.mean(axis=0)
    _, _, directions = np.linalg.svd(centred, full_matrices=False)
    first = directions[:1]
    return centred - (centred @ first.T) @ first

rows = []
for seed in SEEDS:
    texts = sample_documents(seed)
    rng = np.random.default_rng(7 + seed)
    left_random = [texts[i] for i in rng.choice(len(texts), len(pairs))]
    right_random = [texts[i] for i in rng.choice(len(texts), len(pairs))]
    matrix = drop_first_component(skipgram[seed]['center'])
    synonym = cosines(matrix, [p[0] for p in pairs], [p[1] for p in pairs])
    random_pairs = cosines(matrix, left_random, right_random)
    rows.append((synonym.mean(), random_pairs.mean()))
table = np.array(rows)
gap = table[:, 0] - table[:, 1]
print(f"центральна без першої компоненти: синоніми {table[:,0].mean():.4f}, "
      f"випадкові {table[:,1].mean():.4f}, розрив {gap.mean():.4f} ±{gap.std():.4f}")
print(f"середній косинус між усіма словами після цього: "
      f"{average_pairwise_cosine(drop_first_component(skipgram[0]['center'])):.4f}")

## 13 · CBOW: той самий алгоритм навпаки

Skip-gram передбачає сусідів за центральним словом. CBOW робить навпаки: складає
вектори сусідів і передбачає центральне слово. Різниця в одному рядку коду — і в
кількості навчальних прикладів: у CBOW їх рівно стільки, скільки слів у тексті,
бо все вікно йде одним прикладом.

In [ ]:
def build_cbow_arrays(sequences, window):
    """Для кожної позиції — список її сусідів, доповнений нулями до сталої довжини."""
    width = 2 * window
    centers, contexts, lengths = [], [], []
    for sequence in sequences:
        length = len(sequence)
        for position in range(length):
            low, high = max(0, position - window), min(length, position + window + 1)
            around = [int(sequence[j]) for j in range(low, high) if j != position]
            centers.append(int(sequence[position]))
            lengths.append(len(around))
            contexts.append(around + [0] * (width - len(around)))   # добивка, її сховає маска
    return (np.array(centers, dtype=np.int32),
            np.array(contexts, dtype=np.int32),
            np.array(lengths, dtype=np.int32))

cbow_center, cbow_context, cbow_length = build_cbow_arrays(sequences, WINDOW)
print(f"прикладів у CBOW      : {len(cbow_center)}")
print(f"прикладів у skip-gram : {len(pair_center)}")
print(f"skip-gram дорожчий у  : {len(pair_center) / len(cbow_center):.2f} раза")

In [ ]:
class ContinuousBagOfWords(torch.nn.Module):
    """CBOW із тим самим негативним семплюванням, що й у skip-gram."""

    def __init__(self, vocab_size, dim):
        super().__init__()
        self.center = torch.nn.Embedding(vocab_size, dim)     # вектори слів-сусідів
        self.context = torch.nn.Embedding(vocab_size, dim)    # вектори слова-цілі
        torch.nn.init.uniform_(self.center.weight, -0.5 / dim, 0.5 / dim)
        torch.nn.init.zeros_(self.context.weight)

    def scores(self, around, mask, target, negatives):
        vectors = self.center(around) * mask.unsqueeze(2)     # добивку множимо на нуль
        v = vectors.sum(dim=1) / mask.sum(dim=1, keepdim=True)   # середнє по справжніх сусідах
        u_positive = self.context(target)
        u_negative = self.context(negatives)
        positive = (v * u_positive).sum(dim=1)
        negative = torch.bmm(u_negative, v.unsqueeze(2)).squeeze(2)
        return positive, negative


def train_cbow(seed):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    model = ContinuousBagOfWords(len(vocabulary), DIM)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    columns = np.arange(2 * WINDOW)

    started = time.process_time()
    curve, running, steps = [], 0.0, 0
    for epoch in range(EPOCHS):
        order = rng.permutation(len(cbow_center))
        for start in range(0, len(order), BATCH):
            batch = order[start:start + BATCH]
            mask = (columns[None, :] < cbow_length[batch][:, None]).astype(np.float32)
            positive, negative = model.scores(
                torch.from_numpy(cbow_context[batch].astype(np.int64)),
                torch.from_numpy(mask),
                torch.from_numpy(cbow_center[batch].astype(np.int64)),
                torch.from_numpy(draw_negatives(rng, (len(batch), NEGATIVE))))
            loss = sgns_loss(positive, negative)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            running += loss.item(); steps += 1
            if steps % 50 == 0:
                curve.append(running / 50); running = 0.0
    return {'center': model.center.weight.detach().numpy().copy(),
            'context': model.context.weight.detach().numpy().copy(),
            'cpu': time.process_time() - started, 'curve': curve, 'steps': steps}

cbow = {}
for seed in SEEDS:
    cbow[seed] = train_cbow(seed)
    print(f"зерно {seed}: {cbow[seed]['steps']} кроків, процесорних {cbow[seed]['cpu']:.1f} с, "
          f"втрата {cbow[seed]['curve'][-1]:.4f}")

cbow_times = np.array([cbow[s]['cpu'] for s in SEEDS])
print(f"\nCBOW: {cbow_times.mean():.1f} ±{cbow_times.std():.1f} с проти "
      f"{cpu_times.mean():.1f} ±{cpu_times.std():.1f} с у skip-gram — "
      f"швидше у {cpu_times.mean() / cbow_times.mean():.1f} раза")

In [ ]:
unit_cbow = unit_rows(cbow[0]['center'])
print("ті самі слова, але навчені у зворотному напрямку:")
for word in PROBE:
    found = neighbours(unit_cbow, word)
    if found:
        print(f"CBOW · {word:<9} -> " + ' · '.join(f"{w} {c:.3f}" for w, c in found))

Ті самі слова, ті самі сусіди за змістом — але списки різні. Щоб їх можна
було звірити, а не просто подивитись, надрукуймо повні списки з косинусами. Саме ці
числа стоять на карті слів у лекції.

In [ ]:
MAP_WORDS = ['файл', 'помилка', 'вікно', 'мережі', 'пароль', 'кнопка', 'колір', 'звук']
MAP_WORDS = [word for word in MAP_WORDS if word in word_id]
for title, unit in (('skip-gram', unit_center), ('CBOW', unit_cbow)):
    print(f"── {title} ──")
    for word in MAP_WORDS:
        found = neighbours(unit, word, count=8)
        print(f"  {word:<9} " + ' · '.join(f"{w} {c:.3f}" for w, c in found))
    print()

In [ ]:
for seed in SEEDS:
    texts = sample_documents(seed)
    rng = np.random.default_rng(7 + seed)
    left_random = [texts[i] for i in rng.choice(len(texts), len(pairs))]
    right_random = [texts[i] for i in rng.choice(len(texts), len(pairs))]
    for name, matrix in (('CBOW · центральна', cbow[seed]['center']),
                         ('CBOW · сума двох',  cbow[seed]['center'] + cbow[seed]['context'])):
        synonym = cosines(matrix, [p[0] for p in pairs], [p[1] for p in pairs])
        random_pairs = cosines(matrix, left_random, right_random)
        measurements[name].append((synonym.mean(), random_pairs.mean(),
                                   separation(synonym, random_pairs),
                                   100 * np.mean(synonym <= 0)))
        if seed == 0:
            distributions[name] = (synonym, random_pairs)

print(f"{'подання':<24} {'синоніми':>10} {'випадкові':>11} {'РОЗРИВ':>18} {'розділяє':>10}")
for name, rows in measurements.items():
    table = np.array(rows)
    gap = table[:, 0] - table[:, 1]
    print(f"{name:<24} {table[:,0].mean():>10.4f} {table[:,1].mean():>11.4f} "
          f"{gap.mean():>10.4f} ±{gap.std():.4f} {table[:,2].mean():>10.4f}")

Тут видно те, заради чого ми рахували **два** числа замість одного. У CBOW
розрив більший, ніж у skip-gram (0.61 проти 0.54), — але частка випадків, коли
пара синонімів справді перемагає випадкову пару, у skip-gram **вища**.

Отже «розрив середніх» і «чи справді розділяє» — не те саме, і на нашому корпусі
вони показують на різних переможців. Різниця невелика, і чесний висновок такий:
на цьому корпусі два напрямки передбачення дають приблизно однакову якість, а
CBOW при цьому вчиться значно швидше.

## 14 · Де ці вектори підводять

Три межі, і кожну варто побачити числом, а не повірити на слово.

In [ ]:
center = skipgram[0]['center']
lengths = np.linalg.norm(center, axis=1)
correlation = float(np.corrcoef(np.log(np.maximum(word_count, 1)), lengths)[0, 1])
print(f"звʼязок частоти слова й довжини його вектора: кореляція {correlation:.4f}")
print()
print(f"{'слово':<12} {'разів':>8} {'|вектор|':>10}")
for word in ('не', 'для', 'файл', 'каталог', 'пароль'):
    if word in word_id:
        i = word_id[word]
        print(f"{word:<12} {word_count[i]:>8} {lengths[i]:>10.4f}")

In [ ]:
# один вектор на слово: різні значення того самого слова змішуються в одну точку
print("багатозначні слова — усі значення злиті в одну точку:")
for word in ('час', 'вигляд', 'ключа'):
    found = neighbours(unit_center, word, count=6)
    if found:
        print(f"{word:<9} -> " + ' · '.join(w for w, _ in found))
print()
# слова, яких модель не знає взагалі
unknown = [word for word, count in frequency.items() if count < MIN_COUNT]
unknown_uses = sum(frequency[w] for w in unknown)
print(f"слів поза словником моделі: {len(unknown)} — це "
      f"{100 * len(unknown) / len(frequency):.1f} % словоформ,")
print(f"але лише {100 * unknown_uses / total_tokens:.2f} % слововживань")
print("для кожного з них вектора немає взагалі — це і є задача теми 13")

## 15 · Підсумок числами

In [ ]:
print("── корпус ─────────────────────────────────────────────")
print(f"документів                     {len(documents)}")
print(f"слововживань                   {total_tokens}")
print(f"словник моделі (min_count=10)  {len(vocabulary)}, покриття {100*covered/total_tokens:.2f} %")
print(f"пар (слово, сусід) при ±5      {len(pair_center)}")
print(f"сусідів на центр               {len(pair_center)/n_positions:.2f} із 10")
print()
print("── ціна ───────────────────────────────────────────────")
print(f"softmax проти семплювання      у {full_softmax/sampled:.1f} раза дорожче")
print(f"skip-gram, 2 епохи             {cpu_times.mean():.1f} ±{cpu_times.std():.1f} с процесорних")
print(f"CBOW, 2 епохи                  {cbow_times.mean():.1f} ±{cbow_times.std():.1f} с процесорних")
print()
print("── якість ─────────────────────────────────────────────")
for name, rows in measurements.items():
    table = np.array(rows)
    gap = table[:, 0] - table[:, 1]
    print(f"{name:<24} розрив {gap.mean():.4f} ±{gap.std():.4f}, "
          f"розділяє {table[:,2].mean():.4f}")

## 16 · Завдання

**🟢 Рівень 1.** Зміни `WINDOW` з 5 на 2 і навчи skip-gram заново на трьох зернах.
Наведи час, розрив і по чотири сусіди для тих самих слів. Що змінилось більше —
час чи якість?

**🟡 Рівень 2.** Зміни `NEGATIVE` з 5 на 1, 2, 10 і 20. Побудуй таблицю
«негативних → час навчання → розрив ±розкид». Знайди, де приріст якості перестає
перекривати розкид між зернами, і назви це число.

**🔴 Рівень 3.** Реалізуй частотне проріджування (subsampling): викидай слово з
позиції з імовірністю тим більшою, чим воно частіше, і навчи модель на
прорідженому тексті. Заміряй розрив і час на трьох зернах і скажи, чи виграш
більший за розкид.

Повні умови — у [homework.html](homework.html).